# 03 · Evaluation

Benchmarks the fine-tuned model against the base `Qwen 2.5-3B Instruct` on a held-out 50-sample test set.

**Metrics:**
- **Exact Match** — strict syntactic equality (overly strict — semantically equivalent SQL with different aliasing scores 0)
- **BLEU score** — partial credit for n-gram overlap
- **Keyword Overlap** — sanity check that the model picks the right tables/columns
- **Per-complexity breakdown** — exposes where fine-tuning helps most (aggregation queries) vs least (single joins)

**Outputs:** `results/evaluation_results.json` — per-sample base vs fine-tuned outputs + aggregate metrics.

**Prerequisites:** Run `01_data_prep.ipynb` and `02_fine_tuning_full.ipynb` first.


## Install dependencies


In [ ]:
!pip install nltk


## Step 1 · Run base vs fine-tuned evaluation (50 samples)

Loads each model in turn (frees memory between), runs each test sample through both, captures predictions for downstream metric calculation.


In [ ]:
import json
import re
from pathlib import Path
from mlx_lm import load, generate

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

DATA_DIR = Path("data")
BASE_MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
FUSED_DIR = "fused_model"

def mlx_generate(model, tokenizer, user_msg, system=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return generate(model, tokenizer, prompt=prompt, max_tokens=300)

def extract_sql(response_text):
    lines = response_text.strip().split("\n")
    sql_lines = []
    capture = False
    for line in lines:
        if line.strip().upper().startswith("SQL:"):
            capture = True
            sql_lines.append(line.split(":", 1)[1].strip())
        elif line.strip().upper().startswith("EXPLANATION:"):
            capture = False
        elif capture:
            sql_lines.append(line.strip())
    if sql_lines:
        return " ".join(sql_lines).strip().rstrip(";") + ";"
    for line in lines:
        if any(line.strip().upper().startswith(kw) for kw in ["SELECT", "INSERT", "UPDATE", "DELETE", "CREATE", "WITH"]):
            return line.strip()
    return response_text.strip()

def normalize_sql(sql):
    sql = sql.lower().strip().rstrip(";") + ";"
    sql = re.sub(r'\s+', ' ', sql)
    return sql

# Load test data
test_samples = []
with open(DATA_DIR / "test.jsonl") as f:
    for line in f:
        test_samples.append(json.loads(line))

NUM_EVAL = 50

# --- Evaluate Base Model ---
print("Loading base model...")
base_model, base_tokenizer = load(BASE_MODEL_NAME)

base_results = []
print(f"\nEvaluating BASE model on {NUM_EVAL} samples...")
for i, sample in enumerate(test_samples[:NUM_EVAL]):
    user_msg = sample["messages"][1]["content"]
    ground_truth = sample["messages"][2]["content"]
    gt_sql = ""
    for line in ground_truth.split("\n"):
        if line.startswith("SQL:"):
            gt_sql = line.split(":", 1)[1].strip()
            break

    resp = mlx_generate(base_model, base_tokenizer, user_msg)
    pred_sql = extract_sql(resp)
    match = normalize_sql(pred_sql) == normalize_sql(gt_sql)
    base_results.append({
        "match": match, "pred": pred_sql, "gt": gt_sql,
        "question": user_msg, "full_response": resp
    })
    if (i + 1) % 10 == 0:
        correct = sum(r["match"] for r in base_results)
        print(f"  [{i+1}/{NUM_EVAL}] Base accuracy: {correct}/{i+1}")

del base_model, base_tokenizer
print("Base model unloaded.\n")

# --- Evaluate Fine-Tuned Model ---
print("Loading fine-tuned model...")
ft_model, ft_tokenizer = load(FUSED_DIR)

ft_results = []
print(f"\nEvaluating FINE-TUNED model on {NUM_EVAL} samples...")
for i, sample in enumerate(test_samples[:NUM_EVAL]):
    user_msg = sample["messages"][1]["content"]
    ground_truth = sample["messages"][2]["content"]
    gt_sql = ""
    for line in ground_truth.split("\n"):
        if line.startswith("SQL:"):
            gt_sql = line.split(":", 1)[1].strip()
            break

    resp = mlx_generate(ft_model, ft_tokenizer, user_msg)
    pred_sql = extract_sql(resp)
    match = normalize_sql(pred_sql) == normalize_sql(gt_sql)
    ft_results.append({
        "match": match, "pred": pred_sql, "gt": gt_sql,
        "question": user_msg, "full_response": resp
    })
    if (i + 1) % 10 == 0:
        correct = sum(r["match"] for r in ft_results)
        print(f"  [{i+1}/{NUM_EVAL}] Fine-tuned accuracy: {correct}/{i+1}")

del ft_model, ft_tokenizer
print("Fine-tuned model unloaded.\n")

# --- Results Summary ---
base_acc = sum(r["match"] for r in base_results) / NUM_EVAL * 100
ft_acc = sum(r["match"] for r in ft_results) / NUM_EVAL * 100

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Base Model:       {base_acc:.1f}% exact match ({int(base_acc*NUM_EVAL/100)}/{NUM_EVAL})")
print(f"Fine-Tuned Model: {ft_acc:.1f}% exact match ({int(ft_acc*NUM_EVAL/100)}/{NUM_EVAL})")
print(f"Improvement:      +{ft_acc - base_acc:.1f}%")

# Show examples where fine-tuned model improved
print("\n--- Examples Where Fine-Tuned Won ---")
count = 0
for i in range(NUM_EVAL):
    if ft_results[i]["match"] and not base_results[i]["match"] and count < 5:
        count += 1
        print(f"\nExample {count}:")
        print(f"  Question:     {base_results[i]['question'][:150]}...")
        print(f"  Ground Truth: {base_results[i]['gt'][:150]}")
        print(f"  Base Output:  {base_results[i]['pred'][:150]}")
        print(f"  FT Output:    {ft_results[i]['pred'][:150]}")


## Step 2 · Enhanced metrics (BLEU, Keyword Overlap, per-complexity)

Exact Match alone is misleadingly strict for SQL — semantically equivalent queries with different column aliasing or JOIN ordering get scored as failures. BLEU and Keyword Overlap give partial credit and reveal whether the model is at least picking the right tables/columns.


In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

DATA_DIR = Path("data")

# ---- Load test samples with complexity info ----
from datasets import load_dataset
ds = load_dataset("gretelai/synthetic_text_to_sql")
test_indices = ds['test'].shuffle(seed=42).select(range(500, 1000))

# ---- Metrics ----
def normalize_sql(sql):
    sql = sql.lower().strip().rstrip(";") + ";"
    sql = re.sub(r'\s+', ' ', sql)
    return sql

def keyword_overlap(pred, gt):
    """Check if key SQL keywords/structure match."""
    pred_tokens = set(normalize_sql(pred).split())
    gt_tokens = set(normalize_sql(gt).split())
    if not gt_tokens:
        return 0.0
    return len(pred_tokens & gt_tokens) / len(gt_tokens)

def calc_bleu(pred, gt):
    """Calculate BLEU score between predicted and ground truth SQL."""
    ref = normalize_sql(gt).split()
    hyp = normalize_sql(pred).split()
    smooth = SmoothingFunction().method1
    try:
        return sentence_bleu([ref], hyp, smoothing_function=smooth)
    except:
        return 0.0

# ---- Compute enhanced metrics ----
NUM_EVAL = min(50, len(base_results), len(ft_results))

# Per-sample metrics
base_bleu_scores = []
ft_bleu_scores = []
base_keyword_scores = []
ft_keyword_scores = []

# Per-complexity metrics
complexity_results = defaultdict(lambda: {
    "base_exact": 0, "ft_exact": 0,
    "base_bleu": [], "ft_bleu": [],
    "count": 0
})

for i in range(NUM_EVAL):
    gt = base_results[i]["gt"]
    base_pred = base_results[i]["pred"]
    ft_pred = ft_results[i]["pred"]
    complexity = test_indices[i]["sql_complexity"]

    # BLEU
    b_bleu = calc_bleu(base_pred, gt)
    f_bleu = calc_bleu(ft_pred, gt)
    base_bleu_scores.append(b_bleu)
    ft_bleu_scores.append(f_bleu)

    # Keyword overlap
    base_keyword_scores.append(keyword_overlap(base_pred, gt))
    ft_keyword_scores.append(keyword_overlap(ft_pred, gt))

    # By complexity
    c = complexity_results[complexity]
    c["count"] += 1
    c["base_exact"] += int(base_results[i]["match"])
    c["ft_exact"] += int(ft_results[i]["match"])
    c["base_bleu"].append(b_bleu)
    c["ft_bleu"].append(f_bleu)

# ---- Print Results ----
print("=" * 70)
print("COMPREHENSIVE EVALUATION RESULTS")
print("=" * 70)

avg_base_bleu = sum(base_bleu_scores) / len(base_bleu_scores)
avg_ft_bleu = sum(ft_bleu_scores) / len(ft_bleu_scores)
avg_base_kw = sum(base_keyword_scores) / len(base_keyword_scores)
avg_ft_kw = sum(ft_keyword_scores) / len(ft_keyword_scores)
base_exact = sum(r["match"] for r in base_results[:NUM_EVAL])
ft_exact = sum(r["match"] for r in ft_results[:NUM_EVAL])

print(f"\n{'Metric':<25} {'Base Model':>15} {'Fine-Tuned':>15} {'Improvement':>15}")
print("-" * 70)
print(f"{'Exact Match':<25} {base_exact/NUM_EVAL*100:>14.1f}% {ft_exact/NUM_EVAL*100:>14.1f}% {(ft_exact-base_exact)/NUM_EVAL*100:>+14.1f}%")
print(f"{'BLEU Score':<25} {avg_base_bleu:>15.3f} {avg_ft_bleu:>15.3f} {avg_ft_bleu-avg_base_bleu:>+15.3f}")
print(f"{'Keyword Overlap':<25} {avg_base_kw*100:>14.1f}% {avg_ft_kw*100:>14.1f}% {(avg_ft_kw-avg_base_kw)*100:>+14.1f}%")

print(f"\n\n{'='*70}")
print("BREAKDOWN BY SQL COMPLEXITY")
print("=" * 70)
print(f"\n{'Complexity':<25} {'Count':>6} {'Base EM':>10} {'FT EM':>10} {'Base BLEU':>12} {'FT BLEU':>12}")
print("-" * 75)

for complexity in sorted(complexity_results.keys()):
    c = complexity_results[complexity]
    n = c["count"]
    if n == 0:
        continue
    base_em = c["base_exact"] / n * 100
    ft_em = c["ft_exact"] / n * 100
    base_b = sum(c["base_bleu"]) / n
    ft_b = sum(c["ft_bleu"]) / n
    print(f"{complexity:<25} {n:>6} {base_em:>9.1f}% {ft_em:>9.1f}% {base_b:>12.3f} {ft_b:>12.3f}")

# ---- Save enhanced results ----
enhanced_results = {
    "overall": {
        "base_exact_match": base_exact / NUM_EVAL * 100,
        "ft_exact_match": ft_exact / NUM_EVAL * 100,
        "base_bleu": avg_base_bleu,
        "ft_bleu": avg_ft_bleu,
        "base_keyword_overlap": avg_base_kw * 100,
        "ft_keyword_overlap": avg_ft_kw * 100,
    },
    "by_complexity": {
        k: {
            "count": v["count"],
            "base_exact_pct": v["base_exact"] / v["count"] * 100 if v["count"] > 0 else 0,
            "ft_exact_pct": v["ft_exact"] / v["count"] * 100 if v["count"] > 0 else 0,
            "base_bleu": sum(v["base_bleu"]) / v["count"] if v["count"] > 0 else 0,
            "ft_bleu": sum(v["ft_bleu"]) / v["count"] if v["count"] > 0 else 0,
        }
        for k, v in complexity_results.items()
    }
}

with open("eval_results_enhanced.json", "w") as f:
    json.dump(enhanced_results, f, indent=2)

print("\nEnhanced results saved to eval_results_enhanced.json")


## Step 3 · Save full results to JSON

Persists per-sample base vs fine-tuned outputs + aggregate metrics for downstream analysis and the README results table.


In [ ]:
import json

examples = []
for i in range(NUM_EVAL):
    if ft_results[i]["match"] and not base_results[i]["match"] and len(examples) < 5:
        examples.append({
            "question": base_results[i]["question"][:300],
            "ground_truth": base_results[i]["gt"],
            "base_output": base_results[i]["pred"][:300],
            "finetuned_output": ft_results[i]["pred"][:300]
        })

eval_results = {
    "base_accuracy": base_acc,
    "finetuned_accuracy": ft_acc,
    "improvement": ft_acc - base_acc,
    "num_samples": NUM_EVAL,
    "examples": examples
}
with open("../results/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Results saved to ../results/evaluation_results.json")
